# Sesión 12 - Lab 1: Tablas managed y external

Este notebook corre entero sobre el compute serverless habitual del curso. Todo lo que se crea acá vive en un schema descartable (`dbassociate.sesion12_demo`), separado de las tablas reales de sesiones anteriores: se elimina por completo en la celda de Limpieza, sin tocar `dbassociate.bronze`/`silver`/`gold`.

La segunda mitad del notebook (Lab 1C en adelante) necesita un external location real, provisionado una sola vez fuera de este notebook. Los pasos exactos están en la celda de Lab 1C.

## Verificación del entorno

In [ ]:
display(spark.sql("SHOW SCHEMAS IN dbassociate"))
print("Si ves bronze/silver/gold/default en la lista de arriba, el catalog está accesible desde este compute.")

In [ ]:
# Pasamos por widgets para poder cambiar la ruta del external location sin tener que modificar el código

dbutils.widgets.text("external_location_url", "abfss://<tu-contenedor>@<tu-storage-account>.dfs.core.windows.net/sesion12/")
external_location_url = dbutils.widgets.get("external_location_url")

if "<tu-contenedor>" in external_location_url:
    print("Falta completar el widget external_location_url con la ruta real de tu external location (ver prerrequisito en el Lab 1C).")
else:
    print(f"Usando external location: {external_location_url}")

## Lab 1A — Inventario: qué tablas reales existen hoy

Todas las tablas del catálogo `dbassociate` que sobrevivieron a sesiones anteriores (las de las Sesiones 08 y 09, entre otras) son managed: nunca declararon una `LOCATION` propia.

In [ ]:
# Vemos las tablas que tenemos en el catalog dbassociate, para ver si ya se crearon las tablas bronze/silver/gold/default
display(spark.sql("""
SELECT table_catalog, table_schema, table_name, table_type
FROM system.information_schema.tables
WHERE table_catalog = 'dbassociate'
ORDER BY table_schema, table_name
"""))

In [ ]:
display(spark.sql("DESCRIBE EXTENDED dbassociate.silver.clientes"))

Buscá la fila `Location` en la salida de arriba: apunta a una ruta administrada por Unity Catalog (managed storage del metastore, del catalog o del schema), no a una ubicación que alguien haya elegido a mano.

## Lab 1B — Una tabla managed de prueba: DROP y UNDROP

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbassociate.sesion12_demo")

spark.sql("""
CREATE OR REPLACE TABLE dbassociate.sesion12_demo.clientes_demo
AS SELECT * FROM dbassociate.silver.clientes LIMIT 200
""")

display(spark.sql("""
SELECT table_name, table_type
FROM system.information_schema.tables
WHERE table_catalog = 'dbassociate' AND table_schema = 'sesion12_demo'
"""))

In [ ]:
spark.sql("DROP TABLE dbassociate.sesion12_demo.clientes_demo")

try:
    spark.table("dbassociate.sesion12_demo.clientes_demo").count()
except Exception as e:
    print("La tabla ya no existe, como se espera después de un DROP sobre una managed table:")
    print(str(e).splitlines()[0])

spark.sql("UNDROP TABLE dbassociate.sesion12_demo.clientes_demo")

print(f"Filas recuperadas con UNDROP: {spark.table('dbassociate.sesion12_demo.clientes_demo').count()}")

`UNDROP TABLE` solo funciona dentro de una ventana de 7 días desde el `DROP`. Pasado ese plazo, Unity Catalog borra definitivamente los archivos de una managed table eliminada.

## Lab 1C — Prerrequisito: external location real, y la primera tabla external

Antes de este paso hace falta un external location real, creado una sola vez fuera de este notebook:

1. **Azure Portal → Access Connector for Azure Databricks**: crear uno nuevo (o reusar uno existente) y anotar su Resource ID.
2. En la storage account con ADLS Gen2 habilitado: **Access Control (IAM) → Add role assignment → Storage Blob Data Contributor**, asignado a la managed identity del Access Connector.
3. En Databricks, **Catalog → ícono de engranaje → Credentials → Create credential**, tipo Azure Managed Identity, referenciando el Access Connector. Nombre sugerido: `sesion12_storage_credential`.
4. **Catalog → ícono de engranaje → External Locations → Create location**, con URL `abfss://<tu-contenedor>@<tu-storage-account>.dfs.core.windows.net/external_location` y el credential recién creado. Nombre sugerido: `sesion12_external_location`.
5. Completá el widget `external_location_url` de la celda de Verificación del entorno con esa misma URL antes de seguir.

Con eso resuelto, `clientes_demo` (managed, del Lab 1B) sirve de base para crear la primera tabla external de verdad.

In [ ]:
# Creamos una tabla externa a partir de la tabla clientes_demo, para poder ver la diferencia entre managed y external tables
spark.sql(f"""
CREATE OR REPLACE TABLE dbassociate.sesion12_demo.clientes_external_a
LOCATION '{external_location_url}clientes_external_a'
AS SELECT * FROM dbassociate.sesion12_demo.clientes_demo
""")

# validamos la existencia de la tabla externa y managed
display(spark.sql("""
SELECT table_name, table_type
FROM system.information_schema.tables
WHERE table_catalog = 'dbassociate' AND table_schema = 'sesion12_demo'
ORDER BY table_name
"""))

`table_type` debería mostrar `EXTERNAL` para `clientes_external_a` y `MANAGED` para `clientes_demo`: mismo contenido, dos tipos distintos, solo por haber declarado `LOCATION` o no al crearlas.

## Lab 1D — DROP TABLE sobre una external table no borra los archivos

In [ ]:
ruta_clientes_external_a = f"{external_location_url}clientes_external_a"

# comprobamos que los archivos existen en la ruta externa a pesar de realizar el drop.
spark.sql("DROP TABLE dbassociate.sesion12_demo.clientes_external_a")

archivos_restantes = dbutils.fs.ls(ruta_clientes_external_a)
print(f"Archivos que siguen existiendo en la ruta externa después del DROP: {len(archivos_restantes)}")
for archivo in archivos_restantes[:5]:
    print(f"  {archivo.name}")

Los archivos Delta siguen ahí porque Unity Catalog nunca fue dueño del ciclo de vida de esos archivos: el `DROP` solo borró el registro de la tabla en el catálogo, no los datos. Volvamos a registrar una tabla sobre esos mismos archivos, sin volver a escribir nada.

In [ ]:
spark.sql(f"""
CREATE TABLE dbassociate.sesion12_demo.clientes_external_a
LOCATION '{ruta_clientes_external_a}'
""")
# se registran los mismos datos que ya existían en la ruta externa, por lo que 
# no se pierden filas al re-registrar la tabla sobre los mismos archivos.
print(f"Filas visibles después de re-registrar la tabla sobre los mismos archivos: {spark.table('dbassociate.sesion12_demo.clientes_external_a').count()}")

## Lab 1E — Convertir external → managed con SET MANAGED

In [ ]:
%sql
# Convertimos la tabla externa a managed, para poder ver la diferencia entre managed y external tables
# Esta ejecución no borra los archivos de la ruta externa.
# Solo se puede ejecutar con un cluster tipo warehouse
ALTER TABLE dbassociate.sesion12_demo.clientes_external_a SET MANAGED

In [ ]:
display(spark.sql("""
SELECT table_name, table_type
FROM system.information_schema.tables
WHERE table_catalog = 'dbassociate' AND table_schema = 'sesion12_demo' AND table_name = 'clientes_external_a'
"""))

## Lab 1F — UNSET MANAGED: un rollback acotado, no una conversión general

Este comando solo funciona sobre una tabla que se convirtió a managed con `SET MANAGED`, y solo dentro de los 14 días siguientes a esa conversión. No es un comando genérico para pasar cualquier managed table a external: para eso no hay un comando directo (ver Lab 1G).

In [ ]:
%sql
# Este comando solo funciona sobre una que se haya convertido a managed, y lo que hace es volverla a convertir a external.

# Cuando ejecutas el comando ALTER TABLE ... UNSET MANAGED para revertir (hacer un rollback) una tabla de administrada (managed) a externa,
# los archivos de la ruta administrada son los que se eliminan de manera automática.

# Los archivos que estaban en la ruta externa original nunca se borran, ya que el objetivo principal de este comando es que 
# la tabla vuelva a apuntar de forma segura a esos datos externos.

ALTER TABLE dbassociate.sesion12_demo.clientes_external_a UNSET MANAGED

In [ ]:
display(spark.sql("""
SELECT table_name, table_type
FROM system.information_schema.tables
WHERE table_catalog = 'dbassociate' AND table_schema = 'sesion12_demo' AND table_name = 'clientes_external_a'
"""))

## Lab 1G — El camino real de managed nativa a external: CTAS con LOCATION

`clientes_demo` (Lab 1B) nunca fue external: nació managed. Para exportarla a una ubicación externa no hay ningún `ALTER TABLE` que sirva; el camino real es crear una tabla nueva con CTAS, declarando `LOCATION`.

In [ ]:
# creamos una tabla externa a partir de la tabla clientes_demo, para poder ver la diferencia entre managed y external tables
spark.sql(f"""
CREATE OR REPLACE TABLE dbassociate.sesion12_demo.clientes_demo_export
LOCATION '{external_location_url}clientes_demo_export'
AS SELECT * FROM dbassociate.sesion12_demo.clientes_demo
""")

display(spark.sql("""
SELECT table_name, table_type
FROM system.information_schema.tables
WHERE table_catalog = 'dbassociate' AND table_schema = 'sesion12_demo'
ORDER BY table_name
"""))

## Lab 1H — Inspeccionar el external location y sus privilegios

In [ ]:
display(spark.sql("DESCRIBE EXTERNAL LOCATION sesion12_external_location"))

display(spark.sql("""
SELECT *
FROM system.information_schema.external_locations
WHERE external_location_name = 'sesion12_external_location'
"""))

# Asignamos permisos de lectura y creación de tablas externas sobre la external location al grupo curso_ingenieria
# Esto es necesario para que los usuarios del grupo curso_ingenieria puedan crear tablas externas apuntando a esta external location.
spark.sql("GRANT READ FILES, CREATE EXTERNAL TABLE ON EXTERNAL LOCATION sesion12_external_location TO `curso_ingenieria`")

display(spark.sql("""
SELECT grantee, privilege_type
FROM system.information_schema.external_location_privileges
WHERE external_location_name = 'sesion12_external_location'
"""))

`READ FILES` y `CREATE EXTERNAL TABLE` son privilegios del external location, no de ninguna tabla puntual: quien los tenga puede crear o leer cualquier tabla external bajo esa ruta, sin que haga falta un `GRANT` adicional por tabla.

## Limpieza

In [ ]:
spark.sql("REVOKE READ FILES, CREATE EXTERNAL TABLE ON EXTERNAL LOCATION sesion12_external_location FROM `curso_ingenieria`")

for tabla in [
    "dbassociate.sesion12_demo.clientes_demo",
    "dbassociate.sesion12_demo.clientes_external_a",
    "dbassociate.sesion12_demo.clientes_demo_export",
]:
    spark.sql(f"DROP TABLE IF EXISTS {tabla}")

spark.sql("DROP SCHEMA IF EXISTS dbassociate.sesion12_demo")
dbutils.widgets.removeAll()

print("Tablas y schema de este laboratorio eliminados. El external location y su storage credential quedan (son reutilizables entre ediciones).")
print("Los archivos físicos creados en la ruta externa (clientes_external_a, clientes_demo_export) no se borran automáticamente: si querés liberar ese storage, hacelo manualmente desde el explorador de tu cloud.")